# Generate CRML models from natural-language requirements

For each test domain (SRI, Traffic, Pumps) this notebook iterates over the
requirement sequences defined in `experiments.tests.TESTS`, uses an Ollama-backed
LLM with the CRML MCP server to grow the seed model step by step, and writes the
final generated `.crml` file to `generated/`.

Utility functions live in `experiments.util`:
- `extract_crml_block` — pull a CRML code block from LLM output
- `get_req_text`        — normalise requirement text across the heterogeneous test formats
- `generate_crml_sequence` — multi-turn LLM conversation that grows the model

In [1]:
import json
import os
from pathlib import Path

from utils import MultiAgent, create_backend
from experiments.tests import TESTS
from experiments.util import generate_crml_sequence, Tee

# ── Config ────────────────────────────────────────────────────────────────────
OLLAMA_HOST  = "https://demo.narancsle.cc"
AUTH = {
    "CF-Access-Client-Id":     os.environ.get("CF_Access_Client_Id"),
    "CF-Access-Client-Secret": os.environ.get("CF_Access_Client_Secret"),
}
#OLLAMA_HOST = "http://127.0.0.1:11434"  # local Ollama fallback
MCP_CRML_URL = "https://crml-mcp.narancsle.cc/mcp"
K            = 5   # number of independent runs per sequence (best-of-k)

OPTIONS = {
    "qwen3.5:27b" :{
        "folder": "generated/qwen3.5_27b/",
        "endpoint": await create_backend("ollama", "qwen3.5:27b", host=OLLAMA_HOST, headers=AUTH)
    },
    "claude" :{
        "folder": "generated/claude/",
        "endpoint": await create_backend("anthropic", "claude-sonnet-4-5", api_key=os.environ["ANTHROPIC_API_KEY"])
    },
    "openai" :{
        "folder": "generated/openai/",
        "endpoint": await create_backend("openai",    "gpt-5-mini-2025-08-07", api_key=os.environ["OPENAI_API_KEY"])
    }
}
# ─────────────────────────────────────────────────────────────────────────────

settings = OPTIONS["qwen3.5:27b"]

backend = settings["endpoint"]
OUTPUT_DIR   = Path(settings["folder"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Ollama is up. Available models: ['qwen3.5:27b', 'qwen3:14b']
✓ Model 'qwen3.5:27b' is ready.
Anthropic backend ready (model=claude-sonnet-4-5)
OpenAI backend ready (model=gpt-5-mini-2025-08-07)


## SRI domain

In [2]:
for seq_name in TESTS.SRI.keys():
    seq          = TESTS.SRI[seq_name]
    seed         = seq["seed"]
    interactions = seq["interactions"]

    print(f"\n{'='*100}\nSRI / {seq_name}  ({len(interactions)} requirement(s), {K} run(s))\n{'='*100}")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        out_path = OUTPUT_DIR / f"SRI_{seq_name}_k{k}.crml"
        with Tee(out_path.with_suffix(".log")):
            async with MultiAgent([MCP_CRML_URL], backend) as agent:
                results = await generate_crml_sequence(agent, seed, interactions)
                metrics = agent.cumulative_metrics

        out_path.write_text(results[-1] if results else seed)
        out_path.with_suffix(".json").write_text(json.dumps(metrics, indent=2))
        print(f"Saved → {out_path}")


SRI / temp  (2 requirement(s), 5 run(s))

--- run 1/5 ---
User: You are a modelling assistant translating natural language requirements into Common Requirement Modeling Language. I will give you a seed CRML model and then add requirements one by one. After each requirement, extend the model with the CRML formalization of the requirement and return the complete updated model in a ```crml``` code block. The final model must be syntactically valid.

Explain your solution in comments. Tools are available for looking up the CRML coding guidelines, language syntax, as well as for checking model syntax.

Seed model:
```crml
model SRI is FORM_L union {
    Real T is external;
};
```

Acknowledge and wait for the first requirement.
Tools available: ['get_coding_instructions', 'check_syntax', 'list_CRML_resource_files', 'read_hint_file', 'search_hints']

→ Tool call: 'get_coding_instructions' args={}
← Result: [TextContent(type='text', text='# CRML — High-Level Coding Instructions\n\nCRML (Comm

ResponseError: {"type":"https://developers.cloudflare.com/support/troubleshooting/http-status-codes/cloudflare-5xx-errors/error-524/","title":"Error 524: A timeout occurred","status":524,"detail":"The origin web server did not return a complete response within the 120-second Proxy Read Timeout window. The connection was established, but the origin took too long to respond.","instance":"a054f3b80adc8a33","error_code":524,"error_name":"origin_response_timeout","error_category":"origin","ray_id":"a054f3b80adc8a33","timestamp":"2026-06-02T08:07:21Z","zone":"demo.narancsle.cc","cloudflare_error":true,"retryable":true,"retry_after":120,"owner_action_required":true,"what_you_should_do":"**Wait and retry.** Back off for at least 120 seconds. If the error persists, the website operator should check for long-running processes or an overloaded origin.","footer":"This error was generated by Cloudflare on behalf of the website owner."} (status code: 524)

## Traffic-light domain

In [ ]:
for seq_name in TESTS.trafic.keys():
    seq          = TESTS.trafic[seq_name]
    seed         = seq["seed"]
    interactions = seq["interactions"]

    print(f"\n{'='*100}\nTraffic / {seq_name}  ({len(interactions)} requirement(s), {K} run(s))\n{'='*100}")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        out_path = OUTPUT_DIR / f"trafic_{seq_name}_k{k}.crml"
        with Tee(out_path.with_suffix(".log")):
            async with MultiAgent([MCP_CRML_URL], backend) as agent:
                results = await generate_crml_sequence(agent, seed, interactions)
                metrics = agent.cumulative_metrics

        out_path.write_text(results[-1] if results else seed)
        out_path.with_suffix(".json").write_text(json.dumps(metrics, indent=2))
        print(f"Saved → {out_path}")

## Pumping-system domain

In [ ]:
for seq_name in TESTS.pumpsystem.keys():
    seq          = TESTS.pumpsystem[seq_name]
    seed         = seq["seed"]
    interactions = seq["interactions"]

    print(f"\n{'='*100}\nPumps / {seq_name}  ({len(interactions)} requirement(s), {K} run(s))\n{'='*100}")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        out_path = OUTPUT_DIR / f"pumpsystem_{seq_name}_k{k}.crml"
        with Tee(out_path.with_suffix(".log")):
            async with MultiAgent([MCP_CRML_URL], backend) as agent:
                results = await generate_crml_sequence(agent, seed, interactions)
                metrics = agent.cumulative_metrics

        out_path.write_text(results[-1] if results else seed)
        out_path.with_suffix(".json").write_text(json.dumps(metrics, indent=2))
        print(f"Saved → {out_path}")